<a href="https://colab.research.google.com/github/faheemjabbar39-hub/llm-knowledge-integration/blob/main/notebooks/run_baseline_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Baseline run (Base strategy)
**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save. Then run each cell in order (click ▶ or press Shift+Enter).

In [ ]:
# Check that a GPU is attached. You should see 'Tesla T4'.
!nvidia-smi

## 1. Get the code from your private GitHub repo
Paste a GitHub token when asked (github.com/settings/tokens → Generate new token (classic) → tick **repo** → Generate). It is hidden as you type and not saved in the notebook.

In [ ]:
import getpass, os
REPO = "github.com/faheemjabbar39-hub/llm-knowledge-integration.git"
gh_token = getpass.getpass("GitHub token: ")
if not os.path.exists("/content/project"):
    !git clone -q https://{gh_token}@{REPO} /content/project
    !git -C /content/project remote set-url origin https://{REPO}   # don't keep the token on disk
else:
    !git -C /content/project pull -q https://{gh_token}@{REPO} main
del gh_token
%cd /content/project
!git log --oneline -1

## 2. Install packages (~3 minutes)

In [ ]:
!pip install -q -r requirements.txt
!pip install -q -e .
!python -m pytest -q

## 3. Log in to Hugging Face
Paste your Hugging Face **Read** token (starts with `hf_`).

In [ ]:
from huggingface_hub import login
login()

## 4. Download the benchmarks and split them into train / test

In [ ]:
!python scripts/01_prepare_benchmarks.py
!wc -l data/processed/*.jsonl

## 5. Baselines (Base strategy): one cell per model
Each model saves to its **own file** (`results/raw/<model>__base.csv`), so nothing is ever overwritten.
Each cell skips questions that are already answered: running a finished model again just says *0 to run*, and if Colab disconnects mid-run, rerunning the cell continues where it stopped.

Optional: add `--limit 20` to any line for a 2-minute smoke test first.

### 5a. Llama-3.1-8B: ✅ already done (2,000 / 2,000 questions)
No need to run. If you do, it will report 0 to run.

In [ ]:
!python scripts/05_run_eval.py --model llama8b --strategy base

### 5b. Mistral-7B: run this next (~45 min on a T4)

In [ ]:
!python scripts/05_run_eval.py --model mistral7b --strategy base

### 5c. Llama-3.1-70B: needs a large GPU, not the free T4
In 4-bit the 70B model needs about 40 GB of GPU memory. Use an **A100 80GB** (Colab Pro, or a rented GPU on RunPod/Lambda). This cell checks the GPU first and stops safely if it is too small.

In [ ]:
import torch
gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
if gb < 45:
    print(f"GPU has {gb:.0f} GB - too small for the 70B model (needs ~45 GB). Skipping.")
else:
    !python scripts/05_run_eval.py --model llama70b --strategy base

## 6. Results

In [ ]:
!python scripts/06_analyze.py --gpu-usd-per-hour 0.35
!python scripts/08_plot_results.py
import pandas as pd
pd.read_csv('results/summary.csv')

## Charts
One panel per model, one colour per strategy. They grow automatically as you run more models and strategies.

In [ ]:
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('results/figures/*.png')):
    print(f); display(Image(f, width=900))

## 7. Save results to Google Drive (Colab deletes files when the session ends)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/kis_results
!cp -r results/* /content/drive/MyDrive/kis_results/
!ls /content/drive/MyDrive/kis_results/raw

## 8. Upload results to GitHub
Paste your GitHub token again. This saves the results CSVs, summary and charts to the repo with a commit message.

In [ ]:
import getpass, datetime
REPO = "github.com/faheemjabbar39-hub/llm-knowledge-integration.git"
tok = getpass.getpass("GitHub token: ")
!git config user.name "Faheem Jabbar"
!git config user.email "faheemjabbar39@gmail.com"
!git add results/
msg = f"Results update ({datetime.date.today()})"
!git commit -q -m "{msg}" || echo "nothing new to commit"
!git pull -q --rebase -X theirs https://{tok}@{REPO} main
!git push -q https://{tok}@{REPO} HEAD:main && echo "Uploaded to GitHub"
del tok